In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ModuleNotFoundError: No module named 'plotly'

In [ ]:
SUPPLY_CHAIN_PATH = Path("../data/supply_chain.csv")
SUBMISSION_DIR = Path("../submission")
assert SUPPLY_CHAIN_PATH.exists(), f"No existe {SUPPLY_CHAIN_PATH}"
assert SUBMISSION_DIR.exists(), f"No existe {SUBMISSION_DIR}"

In [ ]:
shipments = pd.read_csv(SUPPLY_CHAIN_PATH)
shipments.info()
shipments.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10324 entries, 0 to 10323
Data columns (total 33 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   ID                            10324 non-null  int64  
 1   Project Code                  10324 non-null  object 
 2   PQ #                          10324 non-null  object 
 3   PO / SO #                     10324 non-null  object 
 4   ASN/DN #                      10324 non-null  object 
 5   Country                       10324 non-null  object 
 6   Managed By                    10324 non-null  object 
 7   Fulfill Via                   10324 non-null  object 
 8   Vendor INCO Term              10324 non-null  object 
 9   Shipment Mode                 9964 non-null   object 
 10  PQ First Sent to Client Date  10324 non-null  object 
 11  PO Sent to Vendor Date        10324 non-null  object 
 12  Scheduled Delivery Date       10324 non-null  object 
 13  D

,ID,Project Code,PQ #,PO / SO #,ASN/DN #,Country,Managed By,Fulfill Via,Vendor INCO Term,Shipment Mode,...,Unit of Measure (Per Pack),Line Item Quantity,Line Item Value,Pack Price,Unit Price,Manufacturing Site,First Line Designation,Weight (Kilograms),Freight Cost (USD),Line Item Insurance (USD)
0,1,100-CI-T01,Pre-PQ Process,SCMS-4,ASN-8,Côte d'Ivoire,PMO - US,Direct Drop,EXW,Air,...,30,19,551.0,29.00,0.97,Ranbaxy Fine Chemicals LTD,Yes,13,780.34,NaN
1,3,108-VN-T01,Pre-PQ Process,SCMS-13,ASN-85,Vietnam,PMO - US,Direct Drop,EXW,Air,...,240,1000,6200.0,6.20,0.03,"Aurobindo Unit III, India",Yes,358,4521.5,NaN
2,4,100-CI-T01,Pre-PQ Process,SCMS-20,ASN-14,Côte d'Ivoire,PMO - US,Direct Drop,FCA,Air,...,100,500,40000.0,80.00,0.80,ABBVIE GmbH & Co.KG Wiesbaden,Yes,171,1653.78,NaN
3,15,108-VN-T01,Pre-PQ Process,SCMS-78,ASN-50,Vietnam,PMO - US,Direct Drop,EXW,Air,...,60,31920,127360.8,3.99,0.07,"Ranbaxy, Paonta Shahib, India",Yes,1855,16007.06,NaN
4,16,108-VN-T01,Pre-PQ Process,SCMS-81,ASN-55,Vietnam,PMO - US,Direct Drop,EXW,Air,...,60,38000,121600.0,3.20,0.05,"Aurobindo Unit III, India",Yes,7590,45450.08,NaN


In [ ]:
shipments.loc[0]


ID                                                                              1
Project Code                                                           100-CI-T01
PQ #                                                               Pre-PQ Process
PO / SO #                                                                  SCMS-4
ASN/DN #                                                                    ASN-8
Country                                                             Côte d'Ivoire
Managed By                                                               PMO - US
Fulfill Via                                                           Direct Drop
Vendor INCO Term                                                              EXW
Shipment Mode                                                                 Air
PQ First Sent to Client Date                                       Pre-PQ Process
PO Sent to Vendor Date                                          Date Not Captured
Scheduled Delive

In [ ]:
quality_summary = pd.DataFrame(
    {
        "missing": shipments.isna().sum(),
        "distinct": shipments.nunique(dropna=True),
    }
)
assert shipments["ID"].is_unique, "Cada fila debe representar un envío distinto"
quality_summary.loc[
    [
        "ID",
        "Country",
        "Shipment Mode",
        "Scheduled Delivery Date",
        "Delivered to Client Date",
        "Line Item Value",
        "Freight Cost (USD)",
    ]
]

,missing,distinct
ID,0,10324
Country,0,43
Shipment Mode,360,4
Scheduled Delivery Date,0,2006
Delivered to Client Date,0,2093
Line Item Value,0,8741
Freight Cost (USD),0,6733


In [ ]:
for column in ["Scheduled Delivery Date", "Delivered to Client Date"]:
    shipments[column] = pd.to_datetime(
        shipments[column], format="%d-%b-%y", errors="coerce"
    )

assert (
    shipments[["Scheduled Delivery Date", "Delivered to Client Date"]]
    .notna()
    .all()
    .all()
)
shipments["DeliveryDaysVsSchedule"] = (
    shipments["Delivered to Client Date"] - shipments["Scheduled Delivery Date"]
).dt.days
shipments["IsLate"] = shipments["DeliveryDaysVsSchedule"] > 0
shipments["DeliveryMonth"] = (
    shipments["Delivered to Client Date"].dt.to_period("M").dt.to_timestamp()
)
shipments[
    [
        "Scheduled Delivery Date",
        "Delivered to Client Date",
        "DeliveryDaysVsSchedule",
        "IsLate",
    ]
].head()  

,Scheduled Delivery Date,Delivered to Client Date,DeliveryDaysVsSchedule,IsLate
0,2006-06-02,2006-06-02,0,False
1,2006-11-14,2006-11-14,0,False
2,2006-08-27,2006-08-27,0,False
3,2006-09-01,2006-09-01,0,False
4,2006-08-11,2006-08-11,0,False


In [ ]:

shipments["FreightCostUSD"] = pd.to_numeric(
    shipments["Freight Cost (USD)"], errors="coerce"
)
shipments["HasNumericFreightCost"] = shipments["FreightCostUSD"].notna()
freight_quality = (
    shipments.groupby("HasNumericFreightCost", dropna=False)
    .agg(
        shipments=("ID", "size"),
        line_item_value_usd=("Line Item Value", "sum"),
    )
    .reset_index()
)
freight_quality


,HasNumericFreightCost,shipments,line_item_value_usd
0,False,4126,4.384664e+08
1,True,6198,1.189118e+09


In [ ]:

shipments["FreightCostUSD"] = pd.to_numeric(
    shipments["Freight Cost (USD)"], errors="coerce"
)
shipments["HasNumericFreightCost"] = shipments["FreightCostUSD"].notna()
freight_quality = (
    shipments.groupby("HasNumericFreightCost", dropna=False)
    .agg(
        shipments=("ID", "size"),
        line_item_value_usd=("Line Item Value", "sum"),
    )
    .reset_index()
)
freight_quality

,HasNumericFreightCost,shipments,line_item_value_usd
0,False,4126,4.384664e+08
1,True,6198,1.189118e+09


In [ ]:

overall_kpis = pd.Series(
    {
        "envíos": shipments["ID"].size,
        "valor_total_usd": shipments["Line Item Value"].sum(),
        "cumplimiento_a_tiempo": 1 - shipments["IsLate"].mean(),
        "demora_mediana_días": shipments["DeliveryDaysVsSchedule"].median(),
        "valor_enviado_tarde_usd": shipments.loc[
            shipments["IsLate"], "Line Item Value"
        ].sum(),
    }
)
overall_kpis.to_csv(
    SUBMISSION_DIR / "overall_kpis.csv", index_label="indicador", header=["valor"]
)
overall_kpis

envíos                     1.032400e+04
valor_total_usd            1.627584e+09
cumplimiento_a_tiempo      8.851220e-01
demora_mediana_días        0.000000e+00
valor_enviado_tarde_usd    2.590343e+08
dtype: float64

In [ ]:
px.histogram(
    shipments,
    x="DeliveryDaysVsSchedule",
    nbins=45,
    title="Distribución de días frente a la fecha programada",
    labels={"DeliveryDaysVsSchedule": "Días: real menos programada", "count": "Envíos"},
    color_discrete_sequence=["#1f77b4"],
).add_vline(x=0, line_dash="dash", line_color="#333333")

NameError: name 'px' is not defined

In [ ]:

mode_summary = (
    shipments.groupby("Shipment Mode", dropna=False)
    .agg(
        envíos=("ID", "size"),
        cumplimiento_a_tiempo=("IsLate", lambda value: 1 - value.mean()),
        demora_promedio_días=("DeliveryDaysVsSchedule", "mean"),
        valor_total_usd=("Line Item Value", "sum"),
        valor_enviado_tarde_usd=(
            "Line Item Value",
            lambda value: value[shipments.loc[value.index, "IsLate"]].sum(),
        ),
    )
    .reset_index()
)
mode_summary = mode_summary.rename(
    columns={"Shipment Mode": "modo_de_envío"}
).sort_values("cumplimiento_a_tiempo")
mode_summary.to_csv(SUBMISSION_DIR / "mode_summary.csv", index=False)
mode_summary

In [ ]:
px.bar(
    mode_summary,
    x="cumplimiento_a_tiempo",
    y="modo_de_envío",
    orientation="h",
    text="envíos",
    title="Cumplimiento a tiempo por modo de envío",
    labels={
        "cumplimiento_a_tiempo": "Proporción a tiempo",
        "modo_de_envío": "Modo de envío",
        "envíos": "Envíos",
    },
    color_discrete_sequence=["#f28e2b"],
).update_xaxes(tickformat=".0%")

In [ ]:
minimum_shipments = 50
country_summary = (
    shipments.groupby("Country")
    .agg(
        envíos=("ID", "size"),
        cumplimiento_a_tiempo=("IsLate", lambda value: 1 - value.mean()),
        demora_promedio_días=("DeliveryDaysVsSchedule", "mean"),
        valor_enviado_tarde_usd=(
            "Line Item Value",
            lambda value: value[shipments.loc[value.index, "IsLate"]].sum(),
        ),
    )
    .reset_index()
)
country_summary = country_summary.query("envíos >= @minimum_shipments").sort_values(
    "cumplimiento_a_tiempo"
)
country_summary.to_csv(SUBMISSION_DIR / "country_summary.csv", index=False)
country_summary.head(10)

In [ ]:

priority_countries = country_summary.nlargest(
    10, "valor_enviado_tarde_usd"
).sort_values("valor_enviado_tarde_usd")
priority_countries.to_csv(SUBMISSION_DIR / "priority_countries.csv", index=False)
px.bar(
    priority_countries,
    x="valor_enviado_tarde_usd",
    y="Country",
    orientation="h",
    text="envíos",
    title="Valor expuesto a demora: diez países prioritarios",
    labels={
        "valor_enviado_tarde_usd": "Valor enviado tarde (USD)",
        "Country": "País",
        "envíos": "Envíos",
    },
    color_discrete_sequence=["#e15759"],
)


In [ ]:

country_mode = (
    shipments.groupby(["Country", "Shipment Mode"], dropna=False)
    .agg(
        envíos=("ID", "size"),
        cumplimiento_a_tiempo=("IsLate", lambda value: 1 - value.mean()),
    )
    .reset_index()
    .query("envíos >= 20")
)
country_mode.to_csv(SUBMISSION_DIR / "country_mode_summary.csv", index=False)
country_mode_heatmap = country_mode.pivot(
    index="Country", columns="Shipment Mode", values="cumplimiento_a_tiempo"
)
px.imshow(
    country_mode_heatmap,
    aspect="auto",
    color_continuous_scale="Blues",
    zmin=0,
    zmax=1,
    title="Cumplimiento a tiempo por país y modo de envío (mínimo 20 envíos)",
    labels={"x": "Modo de envío", "y": "País", "color": "Cumplimiento"},
).update_coloraxes(colorbar_tickformat=".0%")

In [ ]:

monthly_summary = (
    shipments.groupby("DeliveryMonth")
    .agg(
        envíos=("ID", "size"),
        cumplimiento_a_tiempo=("IsLate", lambda value: 1 - value.mean()),
    )
    .reset_index()
)
monthly_summary.to_csv(SUBMISSION_DIR / "monthly_summary.csv", index=False)
px.line(
    monthly_summary,
    x="DeliveryMonth",
    y="cumplimiento_a_tiempo",
    markers=True,
    title="Cumplimiento mensual de la fecha prometida",
    labels={
        "DeliveryMonth": "Mes de entrega",
        "cumplimiento_a_tiempo": "Proporción a tiempo",
    },
    color_discrete_sequence=["#4e79a7"],
).update_yaxes(tickformat=".0%")

In [ ]:
priority_segments = (
    shipments.groupby(["Country", "Shipment Mode"], dropna=False)
    .agg(
        envíos=("ID", "size"),
        cumplimiento_a_tiempo=("IsLate", lambda value: 1 - value.mean()),
        valor_enviado_tarde_usd=(
            "Line Item Value",
            lambda value: value[shipments.loc[value.index, "IsLate"]].sum(),
        ),
    )
    .reset_index()
    .query("envíos >= 30")
)
priority_segments.to_csv(SUBMISSION_DIR / "priority_segments.csv", index=False)
priority_segments.sort_values(
    ["valor_enviado_tarde_usd", "cumplimiento_a_tiempo"], ascending=[False, True]
).head(10)

In [ ]:
priority_segments = (
    shipments.groupby(["Country", "Shipment Mode"], dropna=False)
    .agg(
        envíos=("ID", "size"),
        cumplimiento_a_tiempo=("IsLate", lambda value: 1 - value.mean()),
        valor_enviado_tarde_usd=(
            "Line Item Value",
            lambda value: value[shipments.loc[value.index, "IsLate"]].sum(),
        ),
    )
    .reset_index()
    .query("envíos >= 30")
)
priority_segments.to_csv(SUBMISSION_DIR / "priority_segments.csv", index=False)
priority_segments.sort_values(
    ["valor_enviado_tarde_usd", "cumplimiento_a_tiempo"], ascending=[False, True]
).head(10)

In [ ]:

px.bar(
    freight_by_mode.sort_values("flete_sobre_valor_observado"),
    x="flete_sobre_valor_observado",
    y="Shipment Mode",
    orientation="h",
    text="cobertura_valor_con_flete",
    title="Flete como proporción del valor observado, por modo de envío",
    labels={
        "flete_sobre_valor_observado": "Flete / valor con flete disponible",
        "Shipment Mode": "Modo de envío",
        "cobertura_valor_con_flete": "Cobertura",
    },
    color_discrete_sequence=["#59a14f"],
).update_xaxes(tickformat=".0%").update_traces(texttemplate="Cobertura: %{text:.0%}")